# NyxAI 7B QLoRA Fine-Tuning (Free Google Colab / Kaggle)

Fine-tunes `Qwen2.5-7B-Instruct` on your NyxAI Indian female companion + personal assistant dataset (`train.jsonl` + `val.jsonl`) using **Unsloth** (fits on free T4 16 GB GPU + ~12.7 GB RAM) and exports directly to **`Q4_K_M.gguf`** (~4.7 GB) for local Ollama.

### Before you run:
1. **Colab:** `Runtime` -> `Change runtime type` -> **T4 GPU**
2. **Kaggle:** `Settings` -> `Accelerator` -> **GPU T4 x2** (or **P100**) + enable **Internet**

In [ ]:
# 1. Check GPU & Install Unsloth (2-3 minutes)
!nvidia-smi
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Upload train.jsonl and val.jsonl (Colab) or point to Kaggle input
import os

TRAIN_PATH = "train.jsonl"
VAL_PATH = "val.jsonl"

if "COLAB_GPU" in os.environ or os.path.exists("/content"):
    from google.colab import files
    if not os.path.exists(TRAIN_PATH):
        print("Please select training/datasets/processed/train.jsonl and val.jsonl from your PC:")
        uploaded = files.upload()
else:
    # On Kaggle, upload train.jsonl & val.jsonl as a Dataset and update the path below if needed
    for root, _, filenames in os.walk("/kaggle/input"):
        if "train.jsonl" in filenames:
            TRAIN_PATH = os.path.join(root, "train.jsonl")
        if "val.jsonl" in filenames:
            VAL_PATH = os.path.join(root, "val.jsonl")

print("Train file:", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("Val file:  ", VAL_PATH, os.path.exists(VAL_PATH))

In [ ]:
# 3. Load Qwen2.5-7B-Instruct in 4-bit + Attach LoRA Adapters
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Load & Format NyxAI Dataset
from datasets import load_dataset

data_files = {"train": TRAIN_PATH}
if os.path.exists(VAL_PATH):
    data_files["validation"] = VAL_PATH

dataset = load_dataset("json", data_files=data_files)

def format_prompts(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("Train rows:", len(dataset["train"]))
if "validation" in dataset:
    print("Val rows:  ", len(dataset["validation"]))

In [ ]:
# 5. Train QLoRA (3 Epochs ~ 8-15 min on T4)
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

has_val = "validation" in dataset

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"] if has_val else None,
    args=SFTConfig(
        output_dir="nyxai-7b-lora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="epoch" if has_val else "no",
        save_strategy="epoch",
        save_total_limit=1,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        seed=42,
    ),
)

trainer.train()

In [ ]:
# 6. Quick Sanity Test Before Exporting
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are Nyx — a warm, witty, affectionate Indian female AI companion and personal assistant."},
    {"role": "user", "content": "Aaj bahut thak gaya hoon yaar, kuch mood fresh karne ke liye bolo."}
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=120, temperature=0.7, top_p=0.9)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# 7. Export Directly to Q4_K_M GGUF (~4.7 GB) for Ollama
# Note: On Colab, you can also mount Google Drive to save the GGUF so download is fast/resumable!
model.save_pretrained_gguf("nyxai-7b-gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh nyxai-7b-gguf/*.gguf

### After Downloading `*.gguf` to your laptop:
Place the `.gguf` file at `training/experiments/nyxai-7b-q4_k_m.gguf` in your NyxAI repo, then run:
```bash
ollama create nyx-indian-friend-7b -f training/Modelfile.7b.template
```
And set `OLLAMA_MODEL=nyx-indian-friend-7b` in `.env`!